# 06 - Fuentes externas

**Objetivo.** Documentar, cargar y evaluar fuentes externas priorizadas para enriquecer el análisis territorial. Esta etapa separa compatibilidad de integración: primero se verifica cobertura, granularidad y llave de unión.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 180)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'data' / 'raw').exists() and (path / 'README.md').exists():
            return path
    raise FileNotFoundError('No se encontro la raiz del repo. Ejecutar dentro del proyecto.')

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
REPORTS_DIR = PROCESSED_DIR / 'reports'
EXTERNAL_DIR = REPO_ROOT / 'Fuentes_de_enriquecimiento'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)

## Inventario de fuentes externas priorizadas

In [ ]:
fuentes_externas = pd.DataFrame([
    {'fuente': 'Censo 2022 INDEC', 'archivo_o_url': 'Fuentes_de_enriquecimiento/c2022_caba_est_c2_1.xlsx', 'cobertura': 'CABA', 'granularidad': 'Comuna', 'periodo': '2022', 'llave_union': 'comuna', 'aporte': 'Poblacion, superficie y densidad por comuna', 'limitacion': 'No distingue barrios dentro de una misma comuna'},
    {'fuente': 'BA Data - estaciones de subte', 'archivo_o_url': 'URL en datasets_contextuales_caba.py', 'cobertura': 'CABA', 'granularidad': 'Punto/estacion', 'periodo': 'vigente segun fuente', 'llave_union': 'geografica o barrio/comuna derivada', 'aporte': 'Accesibilidad por transporte', 'limitacion': 'Requiere geocodificacion o distancia'},
    {'fuente': 'BA Data - hospitales', 'archivo_o_url': 'URL en datasets_contextuales_caba.py', 'cobertura': 'CABA', 'granularidad': 'Punto/institucion', 'periodo': 'vigente segun fuente', 'llave_union': 'geografica o comuna derivada', 'aporte': 'Entorno de salud', 'limitacion': 'Cantidad no mide calidad ni capacidad'},
    {'fuente': 'BA Data - establecimientos educativos', 'archivo_o_url': 'URL en datasets_contextuales_caba.py', 'cobertura': 'CABA', 'granularidad': 'Punto/institucion', 'periodo': 'vigente segun fuente', 'llave_union': 'geografica o comuna derivada', 'aporte': 'Entorno educativo', 'limitacion': 'Debe distinguir niveles y gestion'},
    {'fuente': 'BA Data - espacios verdes/culturales', 'archivo_o_url': 'URL en datasets_contextuales_caba.py', 'cobertura': 'CABA', 'granularidad': 'Punto/poligono segun dataset', 'periodo': 'vigente segun fuente', 'llave_union': 'geografica o comuna derivada', 'aporte': 'Atributos de entorno urbano', 'limitacion': 'Requiere distancia o agregacion territorial'},
])
display(fuentes_externas)
fuentes_externas.to_csv(PROCESSED_DIR / 'fuentes_externas_inventario_v1.csv', index=False)

## Carga de Censo 2022 por comuna

In [ ]:
ruta_censo = EXTERNAL_DIR / 'c2022_caba_est_c2_1.xlsx'
if not ruta_censo.exists():
    raise FileNotFoundError(f'No se encontro {ruta_censo}')

censo_2022 = pd.read_excel(
    ruta_censo,
    sheet_name='Cuadro 2.1',
    skiprows=5,
    header=None,
    nrows=16,
    usecols='A:E',
    names=['codigo_comuna_indec', 'comuna_texto', 'superficie_km2', 'poblacion_total', 'densidad_hab_km2'],
)
censo_2022_total_ciudad = censo_2022.iloc[[0]].copy()
censo_2022 = censo_2022.iloc[1:].copy()
censo_2022['comuna'] = censo_2022['comuna_texto'].astype(str).str.extract(r'(\d+)').astype(int)
for col in ['superficie_km2', 'poblacion_total', 'densidad_hab_km2']:
    censo_2022[col] = pd.to_numeric(censo_2022[col], errors='coerce')

display(censo_2022)
censo_2022.to_csv(PROCESSED_DIR / 'censo_2022_comunas_v1.csv', index=False)

## Compatibilidad con la base inmobiliaria

In [ ]:
base_path = PROCESSED_DIR / 'df_publicaciones_consolidadas_con_outliers_v1.csv'
if not base_path.exists():
    base_path = PROCESSED_DIR / 'df_publicaciones_consolidadas_v1.csv'

df_base = pd.read_csv(base_path, low_memory=False)
compatibilidad_comuna = pd.DataFrame({
    'metrica': ['filas_totales', 'filas_con_comuna', 'pct_con_comuna', 'comunas_en_base', 'comunas_en_censo'],
    'valor': [
        len(df_base),
        int(df_base['comuna'].notna().sum()) if 'comuna' in df_base.columns else 0,
        round(df_base['comuna'].notna().mean() * 100, 2) if 'comuna' in df_base.columns else 0,
        sorted(df_base['comuna'].dropna().astype(int).unique().tolist()) if 'comuna' in df_base.columns else [],
        sorted(censo_2022['comuna'].dropna().astype(int).unique().tolist()),
    ]
})
display(compatibilidad_comuna)
compatibilidad_comuna.to_csv(REPORTS_DIR / 'compatibilidad_fuentes_externas_comuna.csv', index=False)

## Cierre de fuentes externas

Para la PreEntrega 2 no es obligatorio integrar todas las fuentes. Sí debe quedar claro qué fuente aporta a qué hipótesis, con qué granularidad y qué limitaciones tiene. El Censo 2022 queda listo para cruce por comuna; las fuentes puntuales de transporte, salud, educación y cultura requieren derivar comuna/barrio o calcular distancias.